# 🔗 GENI — Operator: Link Vectorised Document to Bot

**Run this notebook once** after the domain owner has uploaded and processed the ANZSIC CSV.

What this does:
1. Resolves the bot UUID from the bot URL suffix
2. Lists all domain documents available to the bot — finds the one shared by the domain owner
3. Retrieves the `prepared_document_id` (the vectorised/embedded version)
4. Calls `POST /api/operator/bots/{bot_id}/documents/link_unlink` to permanently attach it

Once linked, GENI will RAG-search the embedded CSV whenever the bot receives a question.

### Prerequisites
- `gcloud` CLI installed and authenticated (`gcloud auth login`)
- You hold the `lumi-prod-{domain}-operator` role in MyAccess
- The domain owner has given you either the `document_ids.json` file **or** the `document_name` / `document_id`
- `requests` installed (`pip install requests`)

## 1 · Configuration — fill in your values here

In [ ]:
import json
import subprocess
from pathlib import Path

import requests

# ─────────────────────────────────────────────────────────────────────────────
# ✏️  FILL IN THESE VALUES BEFORE RUNNING
# ─────────────────────────────────────────────────────────────────────────────
GENI_BASE_URL    = "https://eag-lumi-backend-532613543802.australia-southeast1.run.app"
GENI_DOMAIN      = "eag"          # e.g. "eag", "genai", "ambiata"
GCLOUD_PATH      = "gcloud"       # full path if gcloud is not on $PATH

# The bot's URL suffix — the part after /chat/ in the bot's URL.
# E.g. if the bot URL is https://.../chat/anzsic-classifier, set: "anzsic-classifier"
BOT_URL_SUFFIX   = ""             # ← fill in

# From the domain owner's document_ids.json:
#   Option A — load the json file automatically (if placed next to this notebook):
#   Option B — paste the value directly below.
DOCUMENT_NAME    = "anzsic_master_csv"   # must match exactly what the domain owner used
DOCUMENT_ID_HINT = ""                    # optional: paste document_id from domain owner
                                         # to skip the name-lookup step
# ─────────────────────────────────────────────────────────────────────────────

# Auto-load document_ids.json if it exists next to this notebook
_ids_file = Path("document_ids.json")
if _ids_file.exists() and not DOCUMENT_ID_HINT:
    _ids = json.loads(_ids_file.read_text())
    DOCUMENT_NAME    = _ids.get("document_name", DOCUMENT_NAME)
    DOCUMENT_ID_HINT = _ids.get("document_id", "")
    print(f"📂  Loaded document_ids.json: name={DOCUMENT_NAME!r}  id={DOCUMENT_ID_HINT}")

assert BOT_URL_SUFFIX, "BOT_URL_SUFFIX must be set — see comment above"
print(f"Bot URL suffix : {BOT_URL_SUFFIX}")
print(f"Document name  : {DOCUMENT_NAME}")
print(f"Document id    : {DOCUMENT_ID_HINT or '(will search by name)'}")

## 2 · Authentication helper

In [ ]:
def get_token() -> str:
    result = subprocess.run(
        [GCLOUD_PATH, "auth", "print-identity-token"],
        capture_output=True, text=True, check=True, timeout=15,
    )
    return result.stdout.strip()


def make_headers() -> dict:
    return {
        "Authorization": f"Bearer {get_token()}",
        "x-iag-domain":  GENI_DOMAIN,
        "Content-Type":  "application/json",
    }


try:
    tok = get_token()
    print(f"✅  Token obtained (first 20 chars): {tok[:20]}...")
except subprocess.CalledProcessError:
    print("❌  gcloud auth failed — run:  gcloud auth login")
    raise

## 3 · Resolve bot UUID from URL suffix

The link/unlink endpoint needs the bot's **UUID** (`bot_id`), not the `bot_version_id` stored in `.env`.  
We fetch it once here via `GET /api/bots_v2/{bot_url_suffix}`.

In [ ]:
resp = requests.get(
    f"{GENI_BASE_URL}/api/bots_v2/{BOT_URL_SUFFIX}",
    headers=make_headers(),
    timeout=30,
)

if not resp.ok:
    print(f"❌  HTTP {resp.status_code}: {resp.text[:400]}")
    raise RuntimeError("Could not fetch bot info — check BOT_URL_SUFFIX and your role")

bot_info = resp.json()
# The endpoint may wrap the bot under a key; handle both shapes
bot_data = bot_info if "id" in bot_info else next(iter(bot_info.values()), bot_info)
BOT_ID   = bot_data["id"]

print(f"✅  Bot found!")
print(f"    bot_id (UUID)        : {BOT_ID}")
print(f"    bot name             : {bot_data.get('name', 'N/A')}")
print(f"    latest_bot_version_id: {bot_data.get('latest_bot_version', {}).get('id', 'N/A')}")

## 4 · Find the document and get its `prepared_document_id`

`GET /api/operator/bots/{bot_id}/documents` returns all domain documents available to this bot, each with a `prepared_document_id` — the ID of the vectorised/embedded version that the link endpoint requires.

In [ ]:
docs_resp = requests.get(
    f"{GENI_BASE_URL}/api/operator/bots/{BOT_ID}/documents",
    headers=make_headers(),
    timeout=30,
)

if not docs_resp.ok:
    print(f"❌  HTTP {docs_resp.status_code}: {docs_resp.text[:400]}")
    raise RuntimeError("Could not list bot documents")

all_docs = docs_resp.json().get("bot_documents", [])
print(f"Found {len(all_docs)} documents available to this bot:\n")
for d in all_docs:
    linked_flag = "✅ linked" if d.get("linked") else "  unlinked"
    print(f"  {linked_flag}  name={d['name']!r:<30}  prepared_document_id={d['prepared_document_id']}")

In [ ]:
# Match by document_id hint first; fall back to name match
target_doc = None

if DOCUMENT_ID_HINT:
    target_doc = next(
        (d for d in all_docs if d.get("document_id") == DOCUMENT_ID_HINT),
        None,
    )
    if not target_doc:
        print(f"⚠️  No exact document_id match for {DOCUMENT_ID_HINT!r} — trying name match ...")

if not target_doc:
    # Name match (handles the case where the domain owner didn't share the id)
    name_matches = [d for d in all_docs if d["name"] == DOCUMENT_NAME]
    if len(name_matches) == 1:
        target_doc = name_matches[0]
    elif len(name_matches) > 1:
        print(f"⚠️  Multiple documents named {DOCUMENT_NAME!r} — picking the most recently prepared one.")
        target_doc = sorted(name_matches, key=lambda d: d.get("preparation_datetime", ""), reverse=True)[0]

if not target_doc:
    raise RuntimeError(
        f"Document '{DOCUMENT_NAME}' not found in the available documents list.\n"
        "Possible reasons:\n"
        "  • The domain owner's upload is still processing — wait and re-run.\n"
        "  • The document name doesn't match — check DOCUMENT_NAME above.\n"
        "  • The domain owner's document is in a different GENI domain."
    )

PREPARED_DOCUMENT_ID = target_doc["prepared_document_id"]
already_linked       = target_doc.get("linked", False)

print(f"\n✅  Target document found:")
print(json.dumps(target_doc, indent=2))
print(f"\nprepared_document_id : {PREPARED_DOCUMENT_ID}")
print(f"already linked       : {already_linked}")

## 5 · Link the document to the bot

`POST /api/operator/bots/{bot_id}/documents/link_unlink`  
Body: `{ "link": ["<prepared_document_id>"], "unlink": [] }`

In [ ]:
if already_linked:
    print(f"ℹ️  Document is already linked to this bot — nothing to do.")
else:
    link_resp = requests.post(
        f"{GENI_BASE_URL}/api/operator/bots/{BOT_ID}/documents/link_unlink",
        headers=make_headers(),
        json={
            "link":   [PREPARED_DOCUMENT_ID],
            "unlink": [],
        },
        timeout=30,
    )

    if not link_resp.ok:
        print(f"❌  HTTP {link_resp.status_code}: {link_resp.text[:400]}")
        raise RuntimeError("Link call failed")

    result_docs = link_resp.json().get("bot_documents", [])
    linked_doc  = next((d for d in result_docs if d["prepared_document_id"] == PREPARED_DOCUMENT_ID), None)

    if linked_doc and linked_doc.get("linked"):
        print(f"🎉  Document successfully linked to bot!")
        print(json.dumps(linked_doc, indent=2))
    else:
        print("⚠️  Link call succeeded but document not confirmed as linked. Full response:")
        print(json.dumps(link_resp.json(), indent=2))

## 6 · Verify — list currently linked documents

In [ ]:
verify_resp = requests.get(
    f"{GENI_BASE_URL}/api/operator/bots/{BOT_ID}/documents",
    headers=make_headers(),
    timeout=30,
)
verify_resp.raise_for_status()

linked_docs = [d for d in verify_resp.json().get("bot_documents", []) if d.get("linked")]

print(f"Documents currently linked to bot '{BOT_URL_SUFFIX}':\n")
for d in linked_docs:
    print(f"  ✅  name={d['name']!r:<30}  embedding_model={d.get('embedding_model_name', 'N/A')}")
    print(f"      prepared_document_id={d['prepared_document_id']}")
    print(f"      prepared_at={d.get('preparation_datetime', 'N/A')}")
    print()

print("=" * 65)
print("Next step: the ANZSIC classifier bot will now use this document")
print("for RAG-based retrieval on every question. No code changes needed.")
print("=" * 65)